In [1]:
# Sofr data
sfSwRT= [('depo', '1d', 3.60),
      ('swap','1w', 3.66154),('swap','2w', 3.68178),('swap','3w', 3.67395),
      ('swap','1m', 3.6731), ('swap','2m', 3.6794), ('swap','3m', 3.67135),
      ('swap','4m', 3.65985),('swap','5m', 3.65005),('swap','6m', 3.6309),
      ('swap','7m', 3.6101), ('swap','8m', 3.59355),('swap','9m', 3.5725),
      ('swap','10m',3.55115),('swap','11m',3.5339), ('swap','12m',3.5188),
      ('swap','18m',3.43495),('swap','2y', 3.4272), ('swap','3y', 3.4609),
      ('swap','4y', 3.5215), ('swap','5y', 3.58825)]
# Tona data
tnSwRT = [('depo','1d', 0.728),
      ('swap','1w', 0.7277), ('swap','2w', 0.72875),('swap','3w', 0.72875),
      ('swap','1m', 0.72875),('swap','2m', 0.72956),('swap','3m', 0.7414),
      ('swap','4m', 0.76665),('swap','5m', 0.79104),('swap','6m', 0.81747),
      ('swap','7m', 0.84223),('swap','8m', 0.86397),('swap','9m', 0.8872),
      ('swap','10m',0.91146),('swap','11m',0.9338), ('swap','12m',0.9575),
      ('swap','15m',1.02),   ('swap','18m',1.0825), ('swap','2y', 1.2025),
      ('swap','3y', 1.3775), ('swap','4y', 1.51375),('swap','5y', 1.6275)]
# xccy Basis
xcyBSS = [('fxsw','1W',-9.43),('fxsw','1M', -44.85),('fxsw','2M',  -81.51),
        ('fxsw','3M',-121.85),('fxsw','4M',-160.19),('fxsw','5M', -198.75),
        ('fxsw','6M',-234.62),('fxsw','9M',-337.59),('fxsw','12M',-433.70),
        ('bss','18m',-25.625),('bss','2y',-28.125), ('bss','3y',-31.375),
        ('bss','4y', -34.25), ('bss','5y',-36.5)]

In [2]:
from myABBR import * ; import myUTIL as mu
trdDT = jDT(2026,1,20); stlDT = adD(calUJ,trdDT,Tp2); setEvDT(trdDT)
jpySP = 158.1528; print(f'jpySP(T+2):{jpySP:.6f}',end='    jBsCvOBJ(T+0):')
# single & xcBasis(base: sofr)
sfIX, sfCvOBJ, sfCvHDL, sfParRT = mu.makeSofrCurve(sfSwRT)
tnIX, tnCvOBJ, tnCvHDL, tnParRT = mu.makeTonaCurve(tnSwRT)    # baseIX,othrIX,clltCv
jBsCvOBJ,     jBsCvHDL, fxParRT = mu.makeUsdTonaCurve(xcyBSS,jpySP,sfIX,tnIX,sfCvHDL)
mu.showCvDT(jBsCvOBJ);dfDSP(mu.dfNodes(jBsCvOBJ,fxParRT,cmpd=CNT),3)

jpySP(T+2):158.152800    jBsCvOBJ(T+0):tradeDT:2026-01-20, settleDT:2026-01-20 

,date,matYR,parRT,zeroRT,DF
0,2026-01-20,0.0000,nan%,0.601083%,1.00000000
1,2026-01-29,0.0247,-9.430000%,0.601083%,0.99985180
2,2026-02-24,0.0959,-44.850000%,0.578398%,0.99944553
12,2029-01-22,3.0082,-0.313750%,1.057407%,0.96869148
13,2030-01-22,4.0082,-0.342500%,1.165301%,0.95436619
14,2031-01-22,5.0082,-0.365000%,1.257487%,0.93896442


In [3]:
# 1yFxフォワードレート算出                                    ( source:USD target:JPY )
fwMatDT,                uAMT,      payUSf    =\
adY(calUJ, stlDT,1), 1_000_000.0,  False

fwOBJ = ql.FxForward(uAMT,usdCY,jpyCY,jpySP,fwMatDT,payUSf,Tp2,calUJ)
fxENG = ql.DiscountingFxForwardEngine(sfCvHDL,jBsCvHDL,sQH(jpySP));setPE(fwOBJ,fxENG)
fairRT= fwOBJ.fairForwardRate();   spotDT= fwOBJ.settlementDate()
print(f'(QL-ver:{ql.__version__})  tradeDT:{trdDT.ISO()}     '
              f'spotDT:{spotDT.ISO()}     fwMatDT:{fwMatDT.ISO()}')
print(' '*37, f'spotFX:{jpySP:.6f}   fairレート:{fairRT:.6f}'      )

(QL-ver:1.43)  tradeDT:2026-01-20     spotDT:2026-01-22     fwMatDT:2027-01-22
                                      spotFX:158.152800   fairレート:153.815800


In [4]:
# 取引オブジェクト構築       (約定レート:153.85)
fwOBJ = ql.FxForward(uAMT,usdCY,jpyCY,153.85,
                     									fwMatDT,payUSf,Tp2,calUJ); setPE(fwOBJ,fxENG)
uFace = fwOBJ.sourceNominal();     jFace = fwOBJ.targetNominal()
uNPV  = fwOBJ.npvSourceCurrency(); jNPV  = fwOBJ.npvTargetCurrency()
print(f'uFace:{uFace:,.2f}   jFace:{jFace:,.2f}   uNPV:{uNPV:,.2f}  jNPV:{jNPV:,.2f}')

uFace:1,000,000.00   jFace:153,850,000.00   uNPV:-214.64  jNPV:-33,951.86


In [5]:
(153.85-153.8158) * uFace * jBsCvHDL.discount(fwMatDT)

33951.85625884819

In [6]:
# ParFWD: $1mil at a fair forward rate (s:stripの略)
sYR,    sFairRT, sNPVj        =\
[1,2,3],  [],     [];  sMatDT = [adY(calUJ,stlDT,yy) for yy in sYR]
for dd in sMatDT:
	fwOBJ = ql.FxForward(uAMT,usdCY,jpyCY,jpySP,dd,payUSf,Tp2,calUJ);setPE(fwOBJ,fxENG)
	sFairRT.append(fwOBJ.fairForwardRate())                       # 個別fairレートリスト

sDFj = [jBsCvHDL.discount(dd) for dd in sMatDT]
parFX = sum(df*rt for df,rt in zip(sDFj,sFairRT))/sum(sDFj)     # パーレート計算
print(f'parFXレート:{parFX:.6f}   jFace:{uAMT*parFX:,.2f} \n')

for yr, dt, fr in zip(sYR, sMatDT, sFairRT):
	fwOBJ = ql.FxForward(uAMT,usdCY,jpyCY,parFX,dt,payUSf,Tp2,calUJ);setPE(fwOBJ,fxENG)
	sNPVj.append(fwOBJ.npvTargetCurrency())
	print(f'{yr}Y  matDT:{dt.ISO()}    fairFX:{fr:.8f}  jNPV:{sNPVj[-1]:,.2f}')
print(' '*38, f'Total jNPV:{sum(sNPVj):,.4f}')

parFXレート:150.506398   jFace:150,506,398.14 

1Y  matDT:2027-01-22    fairFX:153.81580000  jNPV:3,285,389.95
2Y  matDT:2028-01-24    fairFX:150.42458110  jNPV:-80,325.23
3Y  matDT:2029-01-22    fairFX:147.19774437  jNPV:-3,205,064.72
                                       Total jNPV:-0.0000


In [7]:
(150.50639814-147.19774437) * uFace * jBsCvHDL.discount(jDT(2029,1,22))

3205064.7186863157